# 🧪 Case File 11: Reconstruct the Crime

Welcome to the performance crime scene. This notebook follows a procedure instead of a hunch: capture what Spark actually planned, execute it, collect stage evidence, and only then decide whether WholeStageCodegen deserves attention.

**Mission Objective:** investigate one intentionally imperfect query, demonstrate how a projection can be pruned from `count()`, then force the same expression into a required checksum. Compare the optimized/executed plans, generated-code evidence, and stage metrics before naming the strongest suspect.

**Evidence Guardrail:** benchmark the executed plan, not the DataFrame chain remembered from the source code.


### Step 1: Establish the scene and metrics harvester
AQE remains enabled because production investigations must account for it. We record the setting and query completed stages through the local Spark UI REST API.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import json
import time
from urllib.request import urlopen
from urllib.parse import quote, urlparse

spark = (SparkSession.builder
    .master("local[2]")
    .appName("case-file-11-reconstruct-the-crime")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))

def ui_base_urls(sc):
    url = getattr(sc, "uiWebUrl", None)
    if callable(url):
        url = url()
    if not url:
        return []
    url = url.rstrip("/")
    parsed = urlparse(url)
    return list(dict.fromkeys([url, f"http://127.0.0.1:{parsed.port}"] if parsed.port else [url]))

def completed_stages(spark):
    app_id = quote(spark.sparkContext.applicationId, safe="/")
    path = f"/api/v1/applications/{app_id}/stages?status=complete"
    for base in ui_base_urls(spark.sparkContext):
        try:
            with urlopen(base + path, timeout=10) as response:
                stages = json.loads(response.read().decode("utf-8"))
            latest = {}
            for stage in stages:
                sid = int(stage.get("stageId", -1))
                attempt = int(stage.get("attemptId", 0))
                if sid not in latest or attempt > int(latest[sid].get("attemptId", 0)):
                    latest[sid] = stage
            return list(latest.values())
        except Exception:
            pass
    return []


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:27:42 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:27:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/24 06:27:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


AQE enabled: true
WholeStageCodegen enabled: true


### Step 2: Build the imperfect query once
The expression is expensive enough to be visible, and the repartition plus aggregation gives us a realistic boundary. We first attach the expression to a `count()` action—the suspicious benchmark.


In [2]:
def expensive_expression():
    return (
        F.sqrt(F.col("id") + 1.0)
        + F.log1p(F.col("id") + 1.0)
        + F.sin(F.col("id") * 0.001)
        + F.cos(F.col("id") * 0.002)
    )

def build_query():
    return (spark.range(0, 500_000, 1, numPartitions=2)
        .repartition(8, (F.col("id") % 1000))
        .select(expensive_expression().alias("work"))
        .groupBy()
        .agg(F.sum("work").alias("checksum")))

crime_query = build_query()
print("=== Optimized plan before execution ===")
crime_query.explain("extended")


=== Optimized plan before execution ===


== Parsed Logical Plan ==
'Aggregate ['sum('work) AS checksum#5]
+- Project [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
   +- RepartitionByExpression [(id#0L % cast(1000 as bigint))], 8
      +- Range (0, 500000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
checksum: double
Aggregate [sum(work#4) AS checksum#5]
+- Project [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
   +- RepartitionByExpression [(id#0L % cast(1000 as bigint))], 8
      +- Range (0, 500000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Aggregate [sum(work#4) AS checksum#5]
+- Project [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
   

### Step 3: Catch the pruned benchmark
A projection followed by `count()` can be optimized away because the projected value cannot change row cardinality. We inspect the plan before trusting the timing.


In [3]:
pruned_query = (spark.range(0, 500_000, 1, numPartitions=2)
    .select(expensive_expression().alias("work")))
print("=== Pruned benchmark plan ===")
pruned_query.explain("formatted")
start = time.perf_counter()
pruned_count = pruned_query.count()
pruned_elapsed = time.perf_counter() - start
print(f"pruned count={pruned_count}, elapsed={pruned_elapsed:.3f}s")


=== Pruned benchmark plan ===
== Physical Plan ==
* Project (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#10L]
Arguments: Range (0, 500000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [1]: [(((SQRT((cast(id#10L as double) + 1.0)) + LOG1P((cast(id#10L as double) + 1.0))) + SIN((cast(id#10L as double) * 0.001))) + COS((cast(id#10L as double) * 0.002))) AS work#11]
Input [1]: [id#10L]




pruned count=500000, elapsed=1.346s


### Step 4: Force the work into the result
The corrected benchmark consumes `work` through `sum`. Now the expression affects the returned checksum and cannot be discarded as irrelevant projection work.


In [4]:
before_ids = {int(s.get("stageId", -1)) for s in completed_stages(spark)}
start = time.perf_counter()
forced_result = crime_query.collect()[0]["checksum"]
forced_elapsed = time.perf_counter() - start
time.sleep(1)
after_stages = completed_stages(spark)
forced_stages = [s for s in after_stages if int(s.get("stageId", -1)) not in before_ids]
print(f"forced checksum={forced_result}, elapsed={forced_elapsed:.3f}s, stages={len(forced_stages)}")


forced checksum=241766113.77371216, elapsed=0.987s, stages=3


### Step 5: Inspect what actually ran
The executed plan is closer to CCTV than the source chain. We inspect it after the action and request generated code for the forced query.


In [5]:
print("=== Executed plan ===")
print(crime_query._jdf.queryExecution().executedPlan().toString())
print("=== Formatted executed view ===")
crime_query.explain("formatted")
print("=== Generated-code view ===")
crime_query.explain("codegen")


=== Executed plan ===
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- *(3) HashAggregate(keys=[], functions=[sum(work#4)], output=[checksum#5])
      +- ShuffleQueryStage 1
         +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=100]
            +- *(2) HashAggregate(keys=[], functions=[partial_sum(work#4)], output=[sum#9])
               +- *(2) Project [(((SQRT((cast(id#0L as double) + 1.0)) + LOG1P((cast(id#0L as double) + 1.0))) + SIN((cast(id#0L as double) * 0.001))) + COS((cast(id#0L as double) * 0.002))) AS work#4]
                  +- ShuffleQueryStage 0
                     +- Exchange hashpartitioning((id#0L % 1000), 8), REPARTITION_BY_NUM, [plan_id=71]
                        +- *(1) Range (0, 500000, step=1, splits=2)
+- == Initial Plan ==
   HashAggregate(keys=[], functions=[sum(work#4)], output=[checksum#5])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=19]
      +- HashAggregate(keys=[], functions=[partial_su

### Step 6: Read the stage metrics
We summarize the metrics for the stages created by the forced action. Large shuffle values make data movement a stronger suspect. Little shuffle and spill can shift attention toward local execution, but `executorRunTime` alone is not a direct CPU-time measurement.


In [6]:
forced_metrics = {
    "executorRunTime": sum(
        int(s.get("executorRunTime", 0) or 0) for s in forced_stages
    ),
    "shuffleRead": sum(
        int(s.get("shuffleReadBytes", 0) or 0) for s in forced_stages
    ),
    "shuffleWrite": sum(
        int(s.get("shuffleWriteBytes", 0) or 0) for s in forced_stages
    ),
    "memoryBytesSpilled": sum(
        int(s.get("memoryBytesSpilled", 0) or 0) for s in forced_stages
    ),
    "diskBytesSpilled": sum(
        int(s.get("diskBytesSpilled", 0) or 0) for s in forced_stages
    ),
}

shuffle_bytes = (
    forced_metrics["shuffleRead"] + forced_metrics["shuffleWrite"]
)
spill_bytes = (
    forced_metrics["memoryBytesSpilled"]
    + forced_metrics["diskBytesSpilled"]
)

print("Forced-query stage metrics:", forced_metrics)

if spill_bytes > 0:
    print("Strong suspect: spill/memory pressure.")
    print(
        "Evidence:",
        spill_bytes,
        "spill bytes in completed forced-query stages.",
    )
    print(
        "First test: investigate aggregation memory, partition sizing, "
        "and spill behavior."
    )

elif shuffle_bytes > 0:
    print(
        "Strong suspect: data movement is materially present; "
        "inspect shuffle before tuning codegen."
    )
    print(
        "Evidence:",
        shuffle_bytes,
        "combined shuffle read/write bytes and Exchange nodes "
        "in the executed plan.",
    )
    print(
        "First test: investigate partitioning and the shuffle boundary "
        "before changing codegen thresholds."
    )

else:
    print(
        "Strong suspect: local execution; inspect CPU evidence "
        "and generated code next."
    )
    print(
        "Evidence: no shuffle or spill bytes were reported "
        "for the forced query."
    )
    print(
        "First test: compare warm codegen-on and codegen-off runs "
        "for the executed expression."
    )


Forced-query stage metrics: {'executorRunTime': 1098, 'shuffleRead': 2523023, 'shuffleWrite': 2523023, 'memoryBytesSpilled': 0, 'diskBytesSpilled': 0}
Strong suspect: data movement is materially present; inspect shuffle before tuning codegen.
Evidence: 5046046 combined shuffle read/write bytes and Exchange nodes in the executed plan.
First test: investigate partitioning and the shuffle boundary before changing codegen thresholds.


# 📊 Post-Lab Analysis: Reconstruct the Crime

This investigation began with the executed plan and stage evidence rather than the DataFrame chain. That order matters: Catalyst can prune work, AQE can reshape execution, and generated Java only matters if it sits on a cost path worth investigating.

### 1. The Benchmark Must Execute the Thing

The pruned `select(expensive_expression).count()` plan demonstrates why timing alone is not evidence. If the projection disappears from the optimized plan, the benchmark measured Spark correctly avoiding the intended work.

The forced checksum keeps the expression live and makes the comparison meaningful.

### 2. Plan and Metrics Answer Different Questions

The executed plan shows the operators, exchanges, and codegen regions Spark actually ran. The stage metrics show where to look next: executor runtime, shuffle, spill, or another resource.

### 3. Name the Cause Before Choosing the Fix

Finish the case with a concrete statement: **the strongest suspect is ___; the evidence is ___; therefore the first thing worth testing is ___**. If shuffle is the stronger suspect, investigate partitioning or data movement. If local execution is the stronger suspect, gather CPU evidence and then decide whether WholeStageCodegen is relevant. If spill appears, investigate memory pressure and aggregation shape.

The checklist is the conclusion: never optimize the weapon before identifying the cause of death.
